## Evaluating quantum kernel matrices and solve a simple regression task

This notebook shows how to use sQUlearn to evaluate a fidelity quantum kernel (FQK)

$$
k(x,x')^Q = |\braket{\psi(x)|\psi(x')}|^2
$$

or a projected quantum kernel (PQK) of the form
$$
    k(x,x')^{PQ} = \exp\left(-\gamma\sum_k\sum_{P\in\lbrace X,Y,Z\rbrace}[\mathrm{tr}(P\varrho(x)_k) - \mathrm{tr}(P\varrho(x')_k)]^2\right) = \exp\left(-\gamma[\mathrm{QNN}(x) - \mathrm{QNN}(x')]^2\right)
$$
The quantum encoding circuit/ parameterized quantum circuit to obtain both FQK and QNN is chosen to be the Chebyshev polynomial encoding circuit. The trainable parameters of the quantum encoding circuit are sampled randomly. The training kernel matrix is evaluated for a simple toy function and the corresponding regression task is solved using QKRR. Finally, the results are plotted and the MSE score is evaluated.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from squlearn import Executor
from squlearn.encoding_circuit import ChebyshevTower
from squlearn.kernel.matrix import ProjectedQuantumKernel, KernelOptimizer
from squlearn.kernel.loss.ODE_loss import ODELoss
from squlearn.kernel.ml import QKODE
from squlearn.optimizers.adam import Adam
import sympy as sp


In [2]:
enc_circ = ChebyshevTower(num_qubits=6, num_features=1, num_chebyshev=1, num_layers=6)
pqk = ProjectedQuantumKernel(executor=Executor(), encoding_circuit=enc_circ)

x, f, dfdx, dfdxdx = sp.symbols("x f dfdx dfdxdx")
eq = sp.sin(x) + dfdx + x
initial_values = [1]


loss_ODE_squ = ODELoss(
    eq,
    symbols_involved_in_ODE=[x, f, dfdx],
    initial_values=initial_values,
    boundary_handling="pinned",
)

np.random.seed(0)

x_space = np.linspace(0, 0.9, 25).reshape(-1, 1)

parameter_values = np.random.rand(len(x_space)+1)
labels = np.zeros((len(x_space), 1))

In [3]:
parameter_values

array([0.5488135 , 0.71518937, 0.60276338, 0.54488318, 0.4236548 ,
       0.64589411, 0.43758721, 0.891773  , 0.96366276, 0.38344152,
       0.79172504, 0.52889492, 0.56804456, 0.92559664, 0.07103606,
       0.0871293 , 0.0202184 , 0.83261985, 0.77815675, 0.87001215,
       0.97861834, 0.79915856, 0.46147936, 0.78052918, 0.11827443,
       0.63992102])

In [4]:
optimzer = Adam(options={"maxiter": 100, "lr": 0.1})
kernel_optimizer = KernelOptimizer(quantum_kernel=pqk, loss=loss_ODE_squ, optimizer=optimzer, initial_parameters=parameter_values)

In [10]:
kernel_optimizer.evaluate(x_space.T, labels)

ValueError: Wrong format of an input variable.

In [5]:
kernel_optimizer.run_optimization(x_space, labels)

initial params alpha in kernel optimizer 
  [0.5488135  0.71518937 0.60276338 0.54488318 0.4236548  0.64589411
 0.43758721 0.891773   0.96366276 0.38344152 0.79172504 0.52889492
 0.56804456 0.92559664 0.07103606 0.0871293  0.0202184  0.83261985
 0.77815675 0.87001215 0.97861834 0.79915856 0.46147936 0.78052918
 0.11827443 0.63992102]
26029.77434289379
7744.658886434524
5361.436715777829
8759.413983519436
8699.851684510566
6519.167770693343
4597.506586328723
3523.2533727598525
2957.045588407578
2645.272660591404
2598.66548358687
2846.7547205194446
3213.350361174798
3386.6224948081717
3137.9581176785923
2502.2478582154235
1754.0209028578831
1219.9771883686321
1076.4031030041945
1257.9861506257303
1535.3137959854128
1689.8693696474054
1640.7457139388262
1446.4021372899085
1219.5507078143469
1043.467926539014
946.3007492106057
922.8925480699431
958.9770817735828
1030.762620042614
1093.215872846573
1090.4405138993857
995.574156050658
842.6397689550558
711.5124893477752
672.7166528442312
734

In [6]:
kernel_optimizer.get_optimal_parameters()

array([-3.25427253,  0.69046803,  0.2880992 ,  0.35765733,  0.27779611,
        0.50293376,  0.27232272,  0.70248615,  0.78129253,  0.26251296,
        0.67838346,  0.12609638,  0.63543813,  1.0495957 ,  0.31125054,
        0.43705952,  0.39441573,  1.06668179,  0.61395339,  0.37231692,
        0.80494904,  0.88784274,  0.6616623 ,  0.98094191,  0.16689788,
        0.44814826])

In [7]:
model = QKODE(quantum_kernel=kernel_optimizer)
model.fit(x_space, labels)

AttributeError: 'KernelOptimizer' object has no attribute 'evaluate_derivatives'

In [ ]:



# enc_circ = YZ_CX_EncodingCircuit(num_qubits=4, num_features=4, num_layers=2)
# q_kernel = FidelityKernel(
#     encoding_circuit=enc_circ,
#     executor=Executor("pennylane"),
#     parameter_seed=0,
#     regularization="tikhonov",
# )
# nll_loss = NLL()
# optimzer = Adam(options={"maxiter": 3, "lr": 0.1})
# kernel_optimizer = KernelOptimizer(quantum_kernel=q_kernel, loss=nll_loss, optimizer=optimzer)
# regression_data = make_regression(n_samples=10, n_features=4, random_state=0)

# model = QKRR(quantum_kernel=kernel_optimizer)
# model.fit(regression_data[0], regression_data[1])


initial params alpha in kernel optimizer 
  [ 0.30670429  1.35207467  0.64568133  0.28200936 -0.47969104  0.91667975
 -0.39215112  2.46158236  2.91327904 -0.73235854  1.83296247  0.18155214
  0.42753659  2.67410254 -2.69525994 -2.59414312]
(16,)
(16,)
(16,)
optimla result in kernel optiimzer <squlearn.optimizers.optimizer_base.OptimizerResult object at 0x0000024C98FF49D0>


QKRR(quantum_kernel=<squlearn.kernel.matrix.kernel_optimizer.KernelOptimizer object at 0x0000024C990202E0>)

In [13]:
# class ODELoss():
#     r"""
    
#     Methods:
#     --------
#     """

#     def __init__(self,         
#         ODE_functional=None,
#         symbols_involved_in_ODE=None,
#         initial_values: np.ndarray = None,
#         eta=np.float64(1.0),
#         boundary_handling="pinned",
#         ):
#         super().__init__()
#         self._verify_size_of_ivp_with_order_of_ODE(initial_values, symbols_involved_in_ODE)
#         self.order_of_ODE = (
#             len(symbols_involved_in_ODE) - 2
#         )  # symbols_involved_in_ODE = [x, f, f_, f__, ...]
#         self.symbols_involved_in_ODE = symbols_involved_in_ODE
#         self.ODE_functional = self._create_ODE_loss_format(ODE_functional, symbols_involved_in_ODE)
#         self.initial_values = initial_values
#         self.eta = eta
#         self.boundary_handling = boundary_handling
        
#     def _create_ODE_loss_format(self, ODE_functional, symbols_involved_in_ODE=None):
#         """
#         Given an ODE_functional, returns a function that takes the QNN derivatives list and
#         returns the loss function.

#         Args:
#             ODE_functional (Union[Callable, sympy.Expr]): Functional representation of the ODE
#                                                           (Homogeneous diferential equation).
#                                                           This can be a callable function or a
#                                                           sympy expression. If a sympy expression
#                                                           is given, then, the
#                                                           symbols_involved_in_ODE must be provided.
#             symbols_involved_in_ODE (list): The list of symbols involved in the ODE problem. The
#                                             list of symbols should be in order of differentiation,
#                                             with the first element being the independent variable,
#                                             i.e. [x, f, dfdx, dfdxdx]
#         Returns:
#             QNN_loss (function): The loss function for the QNN with input in the format of the QNN
#                                  tuple derivatives
#         """

#         if isinstance(ODE_functional, sp.Expr):  # if ode_question isinstance of sympy equation
#             if symbols_involved_in_ODE is None:
#                 raise ValueError(
#                     "symbols_involved_in_ODE must be provided"
#                     " if ODE_functional is a sympy equation"
#                 )  # Perhaps this can be somehow improved by list(ODE_functional.free_symbols)
#             _ODE_functional = lambda f_alpha_tensor: sp.lambdify(
#                 symbols_involved_in_ODE, ODE_functional, "numpy"
#             )(*f_alpha_tensor)
#         else:
#             raise ValueError("Only sympy expressions are allowed")

#         return _ODE_functional
    
#     def _verify_size_of_ivp_with_order_of_ODE(self, initial_values, symbols_involved_in_ODE):
#         """
#         Verifies that the length of the initial values vector matches the order of the ODE.

#         Args:
#             initial_values (np.ndarray): Initial values of the ODE
#             order_of_ODE (int): Order of the ODE
#         """
#         order_of_ODE = len(symbols_involved_in_ODE) - 2
#         if order_of_ODE != len(initial_values):
#             raise ValueError(
#                 f"Initial values must have the same length as the order of the ODE. Order of ODE:"
#                 f"{len(symbols_involved_in_ODE)-2},"
#                 f"Length of initial values: {len(initial_values)}"
#             )
#         elif order_of_ODE == 2:
#             print(
#                 "WARNING: 2nd order DEs differentiate the QNN loss function by calculating the"
#                 " second derivative. This can be computationally expensive and inneficient."
#                 " An alternative is to set-up coupled 1rst order DEs (currently not implemented)"
#             )
#         elif order_of_ODE > 2:
#             raise ValueError("Currently, only 1rst and 2nd order ODEs are supported")
    
#     def _get_ODE_functional(self):
#         return self.ODE_functional
    
#     def compute(
#         self,
#         kernel_tensor: np.ndarray,
#         parameter_values: np.ndarray,
#         data: np.ndarray,
#         labels: np.ndarray,
#     ) -> float:
#         """Compute the negative log likelihood loss function.

#         Args:
#             parameter_values (np.ndarray): The parameter values for the variational quantum
#                                            kernel parameters.
#             data (np.ndarray): The training data to be used for the kernel matrix.
#             labels (np.ndarray): The training labels.

#         Returns:
#             float: The negative log likelihood loss value.
#         """

#         # if self._quantum_kernel is None:
#         #     print(
#         #         "Quantum kernel is not set, please set the quantum kernel with set_quantum_kernel method, we are using a precomputed kernel matrix."
#         #     )

#         # else:
#         #     # Bind training parameters
#         #     if self._quantum_kernel.num_parameters > 0:
#         #         raise ValueError(
#         #             "QKODE with a parameterized quantum kernel is not supported yet."
#         #         )
#         #     #TODO implement random parameter generation instead of valueerror
        
#         def f_alpha_order(alpha_, kernel_tensor, order):
#             """Calculates f_alpha.

#             Args:
#                 alpha_ (np.ndarray): The vector of alphas, of shape (len(x_span)+1, 1).
#                 kernel_tensor (tuple): A tuple containing kernel objects for f_alpha_0 and f_alpha_1. 
#                 order (int): Order of the kernel.

#             Returns:
#                 np.ndarray: The vector of f_alphas, of shape (len(x_span), 1).
#             """
#             alpha = alpha_[1:]
#             if order == 0:
#                 return np.dot(kernel_tensor[order], alpha) + alpha_[0]
#             return np.dot(kernel_tensor[order], alpha) 

#         if kernel_tensor is None:
#             kernel_tensor = [self._quantum_kernel.evaluate_derivatives(data, values = "K")["K"], 
#                              self._quantum_kernel.evaluate_derivatives(data, values = "dKdx")["dKdx"]]
#             if self.order_of_ODE > 2:
#                 kernel_tensor.append(self._quantum_kernel.evaluate_derivatives(data, values = "dKdxdx")["dKdxdx"])


#         f_alpha_tensor = np.array([f_alpha_order(parameter_values, kernel_tensor, i) for i in range(self.order_of_ODE+1)])

        
#         sum1 = np.sum((self.ODE_functional([data, *f_alpha_tensor])**2)) #Functional
#         sum2 = np.sum((f_alpha_tensor[:,0][:len(self.initial_values)] - self.initial_values)**2) #Initial condition
#         L = sum2 + sum1 * self.eta

        
#         return L